# Этап 5. Обучение и оценка моделей
Запускайте из корня проекта после этапа 4. Нужны pandas, numpy, scikit-learn и lightgbm. Если пакет отсутствует, выполните в отдельной ячейке `%pip install lightgbm==4.6.0 scikit-learn` и перезапустите ядро. Число деревьев выбирается только на валидации. После выбора полная модель обучается на двух периодах. Кандидаты оцениваются без подвыборки.

In [ ]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import lightgbm as lgb
import sklearn
from IPython.display import display

source = Path("results_stage4")
out = Path("results_stage5")
out.mkdir(exist_ok=True)
groups = json.loads((source / "feature_groups.json").read_text(encoding="utf-8"))
features = [field for fields in groups.values() for field in fields]
summary = pd.read_csv(source / "feature_summary.csv").set_index("период")
metrics = ["Precision@10", "Recall@10", "NDCG@10"]
k = 10

def read_split(stem, name):
    frame = pd.read_csv(source / f"{stem}.csv.gz").sort_values(["user", "item"]).reset_index(drop=True)
    sizes = frame.groupby("user").label.agg(["size", "sum"])
    observed = [len(frame), len(sizes), int(frame.label.sum())]
    reference = summary.loc[name, ["кандидатные пары", "группы", "положительные пары"]].tolist()
    if observed != reference or frame.duplicated(["user", "item"]).any():
        raise ValueError(f"{name}: размеры или ключи расходятся с данными этапа 4")
    if not (frame.label.isin([0, 1]).all() and sizes["size"].ge(k).all()
            and sizes["sum"].between(1, sizes["size"] - 1).all()
            and np.isfinite(frame[features].to_numpy()).all()):
        raise ValueError(f"{name}: проверьте признаки, метки и размеры групп")
    return frame

train = read_split("train", "Обучение")
valid = read_split("validation", "Валидация")

## Общая оценка списков
Сортировка выполняется по убыванию балла и возрастанию идентификатора фильма. Для Recall и идеального DCG подсчитываются все релевантные кандидаты пользователя. Итог — среднее по пользователям.

In [ ]:
discount = 1 / np.log2(np.arange(2, k + 2))
ideal = np.r_[0.0, np.cumsum(discount)]

def evaluate(frame, scores):
    if len(scores) != len(frame) or not np.isfinite(scores).all():
        raise ValueError("Проверьте предсказания")
    ordered = frame[["user", "item", "label"]].assign(score=np.asarray(scores)).sort_values(
        ["user", "score", "item"], ascending=[True, False, True])
    top = ordered.groupby("user", sort=False).head(k).copy()
    top["rank"] = top.groupby("user").cumcount() + 1
    top["gain"] = top.label * discount[top["rank"].to_numpy() - 1]
    by_user = top.groupby("user").agg(hits=("label", "sum"), dcg=("gain", "sum"))
    by_user["relevant"] = frame.groupby("user").label.sum()
    by_user["Precision@10"] = by_user.hits / k
    by_user["Recall@10"] = by_user.hits / by_user.relevant
    by_user["NDCG@10"] = by_user.dcg / ideal[np.minimum(k, by_user.relevant.to_numpy())]
    return by_user.reset_index(), top.drop(columns="gain")

# Контроль на условном списке: один релевантный фильм на второй позиции.
toy = pd.DataFrame({"user": 1, "item": range(12), "label": [0, 1] + [0]*10})
check, _ = evaluate(toy, np.zeros(len(toy)))
np.testing.assert_allclose(check[metrics].iloc[0], [0.1, 1, 1 / np.log2(3)])

## Выбор числа деревьев
Обучаем один ансамбль и сравниваем его первые 30, 60 и 100 деревьев. При равном NDCG выбираем меньшее число. В API LightGBM group — последовательность размеров смежных групп строк ([документация](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRanker.html)). При объединении периодов сохраняем отдельную группу пользователя на каждую дату.

In [ ]:
grid = [30, 60, 100]
params = dict(objective="lambdarank", learning_rate=0.05, num_leaves=15,
              min_child_samples=30, lambdarank_truncation_level=13, random_state=42,
              deterministic=True, force_col_wise=True, n_jobs=2, verbosity=-1)

def fit_ranker(frames, trees):
    merged = pd.concat(frames, ignore_index=True)
    sizes = np.concatenate([part.groupby("user", sort=False).size().to_numpy()
                            for part in frames])
    model = lgb.LGBMRanker(n_estimators=trees, **params)
    model.fit(merged[features], merged.label.to_numpy(dtype=np.float32, copy=True), group=sizes)
    return model

trial = fit_ranker([train], max(grid))
records = []
for trees in grid:
    scores = trial.predict(valid[features], num_iteration=trees)
    values, _ = evaluate(valid, scores)
    records.append({"trees": trees, **values[metrics].mean().to_dict()})
selection = pd.DataFrame(records)
best = int(selection.sort_values(["NDCG@10", "trees"], ascending=[False, True]).iloc[0]["trees"])
selection.to_csv(out / "validation_grid.csv", index=False, encoding="utf-8-sig")
display(selection)
print("Выбрано деревьев:", best)

## Переобучение и однократная тестовая оценка
Популярность — число прошлых оценок 4–5. Жанровая модель использует косинусное сходство, а при отсутствии положительного профиля — популярность. Метрики всех моделей вычисляются одной функцией.

In [ ]:
final_model = fit_ranker([train, valid], best)
test = read_split("test", "Тест")
popularity = test.item_positive_count.to_numpy()
genre_score = np.where(test.user_positive_share.gt(0), test.genre_cosine, popularity)
predictions = {"Популярность": popularity, "Жанровое сходство": genre_score,
               "LightGBM Ranker": final_model.predict(test[features])}
user_tables, top_tables = [], []
for model_name, scores in predictions.items():
    values, top = evaluate(test, scores)
    user_tables.append(values.assign(model=model_name))
    top_tables.append(top.assign(model=model_name))
per_user = pd.concat(user_tables, ignore_index=True)
recommendations = pd.concat(top_tables, ignore_index=True)
comparison = per_user.groupby("model", sort=False)[metrics].mean()
comparison.insert(0, "users", per_user.groupby("model").user.nunique())
comparison.to_csv(out / "test_metrics.csv", encoding="utf-8-sig")
per_user.to_csv(out / "per_user.csv", index=False, encoding="utf-8-sig")
recommendations.to_csv(out / "top10.csv", index=False, encoding="utf-8-sig")
run = {"best_trees": best, "grid": grid, "params": params, "features": features,
       "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "lightgbm": lgb.__version__, "sklearn": sklearn.__version__}
(out / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
display(comparison)
print("Результаты:", out.resolve())